# ICS 604: APPLIED DATA SCIENCE 

## Kernel Density Estimation Bandwidth

---

In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

## Optimal Kernel Width

The kernel in KDE is defined by its **width**, commonly referred to as the **bandwidth** in programming libraries. The bandwidth controls the scale of the kernel and applies regardless of which kernel function we choose (e.g., tophat, exponential, Gaussian, cosine, etc.). While the kernel shape determines *how* neighboring points are weighted, the bandwidth determines *how far* that influence extends.

A natural question arises: **how do we choose an appropriate bandwidth?**

In a previous example, we set the bandwidth (`bw`) to `0.16`. That value was chosen as a reasonable smoothing parameter for the scale of the data — but in practice, bandwidth selection is not arbitrary. It directly determines how much neighboring observations contribute to the estimated density.

Recall:
- **Large bandwidth (high scale)** $\rightarrow$ distant neighbors significantly influence the estimate.
- **Small bandwidth (low scale)** $\rightarrow$ only very close neighbors contribute meaningfully.

In [ ]:
fig = plt.figure(figsize=(10, 4))
kernel_1 = [sp.stats.norm.pdf(x, -3, 2) for x in np.arange(-10, 10, 0.1)]
kernel_2 = [sp.stats.norm.pdf(x, 0, 1) for x in np.arange(-10, 10, 0.1)]
kernel_3 = [sp.stats.norm.pdf(x, 3, 0.5) for x in np.arange(-10, 10, 0.1)]

plt.plot(np.arange(-10, 10, 0.1), kernel_1)
plt.plot(np.arange(-10, 10, 0.1), kernel_2)
plt.plot(np.arange(-10, 10, 0.1), kernel_3);

Choosing the correct bandwidth is crucial. Poor choices lead to misleading density estimates:

**If the bandwidth is too small**
- The KDE **under-smooths** the data.
- The estimate becomes very jagged and highly sensitive to small fluctuations in the sample.
- This corresponds to **overfitting** in machine learning: the model captures noise rather than the underlying distribution.

In [ ]:
x_mean, x_scale =  0, 0.5
y_mean, y_scale =  9, 1

x_data = np.random.normal(x_mean, x_scale, 100)
y_data = np.random.normal(y_mean, y_scale, 100)

all_data = np.concatenate([x_data, y_data])  # list_1 + list_2  
print(len(all_data))

In [ ]:
x_values = np.arange(-10, 20, 0.25)

kde = sp.stats.gaussian_kde(all_data, bw_method=0.02)
densities = kde.evaluate(x_values)

fig = plt.figure(figsize=(10, 4))
plt.hist(all_data, bins=50, density=True, edgecolor="k", linewidth=1, alpha=0.2)
plt.plot(x_values, densities, lw=2, color='r');

**If the bandwidth is too large**
- The KDE **over-smooths** the data.
- Peaks are flattened and important structure in the data may disappear.
- This corresponds to **underfitting** in machine learning: the model is too simple to capture meaningful patterns.

In [ ]:
kde = sp.stats.gaussian_kde(all_data, bw_method=0.6)
densities = kde.evaluate(x_values)

kde = sp.stats.gaussian_kde(all_data, bw_method=1)
densities_2 = kde.evaluate(x_values)

plt.figure(figsize=(10, 4))
plt.hist(all_data, bins=50, density=True, edgecolor="k", linewidth=1, alpha=0.5)
plt.plot(x_values, densities, lw=2, color='black', label="bandwidth = 0.6")
plt.plot(x_values, densities_2, lw=2, color='red', label="bandwidth = 1")
plt.legend();

Thus, bandwidth controls the fundamental **bias–variance tradeoff**:

- Small bandwidth $\rightarrow$ low bias, high variance
- Large bandwidth $\rightarrow$ high bias, low variance

Selecting an appropriate bandwidth is often more important than choosing the specific kernel function. 

### How Do We Choose the Correct Value of the Bandwidth?

When applying KDE, our objective is to estimate the true population probability density function (pdf) from a finite sample. Because the dataset is limited, the estimate will never perfectly match the true distribution. The goal is therefore to choose a bandwidth that makes the estimated density *as close as possible to the unknown population pdf*.

Conceptually, we want a bandwidth that is:

- **Narrow enough** to preserve important structural details in the data
- **Wide enough** to avoid excessive variability and spurious fluctuations

This tension reflects a fundamental principle in statistics and machine learning: the **Bias–Variance Tradeoff**.

- **Small bandwidth**: The estimator follows the data very closely. While this captures fine details, it may also model random noise (overfitting).
- **Large bandwidth**: The estimator is smoother and more stable, but may miss meaningful features of the true distribution (underfitting).

Importantly, bandwidth selection is typically far more influential than the specific choice of kernel. A well-chosen bandwidth with a simple kernel often outperforms a poorly chosen bandwidth with a sophisticated kernel. Ultimately, selecting the correct bandwidth is about balancing smoothness and fidelity to the data — achieving the best compromise between bias and variance.

## Illustrating the Bias-Variance Tradeoff

To demonstrate the bias–variance tradeoff in KDE, suppose we repeatedly generate datasets from the same underlying population. Consider a mixture of two Gaussian distributions:
$$
X \sim \mathcal{N}(0, 1) \text{, \quad} Y \sim \mathcal{N}(9, 1)
$$
  
Each simulated dataset is drawn from this same population structure. Because the true population distribution is fixed, we would ideally expect the estimated densities from different samples to look very similar to one another — and to the true underlying density.

In [ ]:
plt.figure(figsize=(10, 4))
x_values = np.arange(-10, 20, 0.1)
x_mean, x_scale = 0, 1
y_mean, y_scale = 9, 1

colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'w']
for i in range(len(colors)):
    x_data = np.random.normal(x_mean, x_scale, 50)
    y_data = np.random.normal(y_mean, y_scale, 50)
    data = np.concatenate([x_data, y_data])
    kde = sp.stats.gaussian_kde(data, bw_method=0.4)
    densities = kde.evaluate(x_values)
    plt.plot(x_values, densities, lw=2, color=colors[i])

### Model Variance

In this context, **variance** refers to the variability in the model’s output that arises from small fluctuations in the training data. If we draw multiple datasets from the same underlying population and fit the model separately on each one, a high-variance model will produce noticeably different results across those samples.

A model with **high variance** focuses too closely on the specific observations in the training data. It captures not only the true underlying structure but also random noise and incidental patterns that happen to appear in that particular sample. As a result, the model does not generalize well to unseen data.

Such models typically:
- Perform very well on training data
- Exhibit large changes when the training dataset is slightly modified
- Produce high error on new (test) data

In the context of KDE, choosing a **very small bandwidth** leads to high variance. The estimated density becomes highly sensitive to individual data points, resulting in a jagged curve that differs substantially across samples drawn from the same population. Thus, high variance corresponds to **overfitting**: the model learns the noise instead of the true underlying distribution.

In [ ]:
plt.figure(figsize=(10, 4))
x_values = np.arange(-10, 20, 0.1)
x_mean, x_scale = 0, 1
y_mean, y_scale = 9, 1

colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'w']
for i in range(len(colors)):
    x_data = np.random.normal(x_mean, x_scale, 50)
    y_data = np.random.normal(y_mean, y_scale, 50)
    data = np.concatenate([x_data, y_data])
    kde = sp.stats.gaussian_kde(data, bw_method=0.05)
    densities = kde.evaluate(x_values)
    plt.plot(x_values, densities, lw=2, color=colors[i])

### Model Bias

In this context, **bias** refers to the systematic difference between the estimated density and the true underlying population density. A model with high bias makes strong simplifying assumptions that prevent it from accurately capturing the true structure of the data.

**High-bias** models tend to **oversimplify** the estimated density. They smooth over important features, such as multiple peaks or asymmetries, and produce curves that may look clean and stable but fail to reflect the actual data-generating process.

When bias is high:
- The contribution of individual data points is minimized.
- The estimated model changes very little even if the data changes.
- Important patterns in the data may be ignored.
- Both training and test errors can be large due to systematic underfitting.

In the context of KDE, choosing a **very large bandwidth** leads to high bias. The smoothing is so strong that distinct features (e.g., multiple modes) may disappear, and the estimated density becomes overly flat. Thus, high bias corresponds to **underfitting**: the model is too simple to capture the true structure of the underlying distribution.

In [ ]:
plt.figure(figsize=(10, 4))
x_values = np.arange(-10, 20, 0.1)
x_mean, x_scale = 0, 1
y_mean, y_scale = 9, 1

colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'w']
for i in range(len(colors)):
    x_data = np.random.normal(x_mean, x_scale, 50)
    y_data = np.random.normal(y_mean, y_scale, 50)
    data = np.concatenate([x_data, y_data])
    kde = sp.stats.gaussian_kde(data, bw_method=1)
    densities = kde.evaluate(x_values)
    plt.plot(x_values, densities, lw=2, color=colors[i])

In [ ]:
plt.figure(figsize=(10 ,4))
x_values = np.arange(-10, 20, 0.1)
x_mean, x_scale = 0, 1
y_mean, y_scale = 9, 1

colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'w']
for i in range(len(colors)):
    x_data = np.random.normal(x_mean, x_scale, 50)
    y_data = np.random.normal(y_mean, y_scale, 50)
    data = np.concatenate([x_data, y_data])
    kde = sp.stats.gaussian_kde(data, bw_method=2)
    densities = kde.evaluate(x_values)
    plt.plot(x_values, densities, lw=2, color=colors[i])
    
x_mean, x_scale = -1, 1
y_mean, y_scale = 8, 1

for i in range(len(colors)):
    x_data = np.random.normal(x_mean, x_scale, 50)
    y_data = np.random.normal(y_mean, y_scale, 50)
    data = np.concatenate([x_data, y_data])
    kde = sp.stats.gaussian_kde(data, bw_method=2)
    densities = kde.evaluate(x_values)
    plt.plot(x_values, densities, lw=2, color=colors[i])

### Model Variance and Bias Revisited 

From the previous discussion, we can summarize the key ideas:
- High-variance models produce estimates that fluctuate significantly across different samples from the same population. Their predictions are highly sensitive to small changes in the data.
- High-bias models produce systematically distorted estimates. The model form is overly restrictive, resulting in a curve that consistently misses important structure in the data.

In KDE:
- Small bandwidth $\rightarrow$ high variance (overfitting, overly wiggly curve)
- Large bandwidth $\rightarrow$ high bias (underfitting, overly smooth curve)

Our objective is to choose a bandwidth that best balances these two effects — one that produces an estimate with minimal bias and minimal variance. In practice, methods such as rule-of-thumb formulas (e.g., Silverman’s rule), cross-validation, or plug-in estimators are commonly used to determine a near-optimal bandwidth automatically.

### Selecting the Kernel Bandwidth: Theoretical

There are two broad approaches to selecting the kernel bandwidth: theoretical (analytic) and empirical (data-driven).

The theoretical approach estimates the optimal bandwidth using a closed-form expression derived under certain assumptions about the underlying distribution. A classic example is Silverman’s rule of thumb, which is implemented in scipy’s `gaussian_kde`.

The rule is:

$$
\large
h = 0.9\min\left(\hat{\sigma},\frac{IQR}{1.34}\right)n^{-\frac{1}{5}}
$$

where:
- $\hat{\sigma}$ is the standard deviation of the data
- $IQR$ is the interquartile range (the difference between the 75th and 25th percentiles)
- $n$ is the dataset size

This formula is derived by minimizing an asymptotic approximation of the mean integrated squared error (MISE), assuming the underlying distribution is approximately Gaussian.

Advantages:
- Extremely fast
- No iteration required
- Widely used in classical statistical analysis

You simply plug in the numbers and obtain a reasonable default bandwidth.

### Selecting the Kernel Bandwidth: Empirical

The empirical approach relies directly on the data rather than a closed-form formula.

The idea is:
1. Choose a candidate bandwidth.
2. Fit the KDE on a subset of the data.
3. Evaluate how well the fitted KDE explains the remaining data.
4. Repeat for different bandwidth values.
5. Select the bandwidth that performs best.

This procedure is known as **cross-validation**.

Unlike theoretical rules, cross-validation does not rely on strong distributional assumptions. Instead, it selects the bandwidth that best generalizes to unseen data. While more computationally intensive, it often yields better results when the underlying distribution deviates from Gaussian assumptions.

This approach is commonly discussed in greater depth in machine learning contexts.

### Selecting the Kernel Bandwidth: Practical

In practice, especially for visualization purposes, a hybrid approach is often sufficient.

A common strategy is:
- Start with the **default bandwidth** suggested by a theoretical rule (e.g., Silverman’s rule).
- Then **manually adjust** the bandwidth slightly to produce the most intuitive and interpretable visualization.

For example, if we know a priori that the data come from two distinct distributions, we may prefer a bandwidth that preserves two visible peaks rather than smoothing them into one. In such cases, prior knowledge about the data-generating process can guide the final choice.

Thus, bandwidth selection is not purely mechanical — it often combines theoretical guidance, empirical validation, and domain knowledge to achieve a meaningful density estimate.

In [ ]:
x_values = np.arange(-10, 20, 0.25)
kde = sp.stats.gaussian_kde(all_data)  # bw_method='scott' by default
densities = kde.evaluate(x_values)

plt.figure(figsize=(10, 4))
plt.hist(all_data, bins=50, edgecolor="k", linewidth=1, density=True, alpha=0.5)
plt.plot(x_values, densities, lw=2, color='black');

In [ ]:
x_values = np.arange(-10, 20, 0.25)
kde = sp.stats.gaussian_kde(all_data, bw_method='silverman')
densities = kde.evaluate(x_values)

plt.figure(figsize=(10, 4))
plt.hist(all_data, bins=50, edgecolor="k", linewidth=1, density=True, alpha=0.5)
plt.plot(x_values, densities, lw=2, color='black');

In [ ]:
x_values = np.arange(-10, 20, 0.25)
kde = sp.stats.gaussian_kde(all_data, bw_method=0.09)
densities = kde.evaluate(x_values)

plt.figure(figsize=(10, 4))
plt.hist(all_data, bins=50, edgecolor="k", linewidth=1, density=True, alpha=0.5)
plt.plot(x_values, densities, lw=2, color='black');